# Simple Model from Brain Atlas Dataset

This notebook builds a simple model with two types of neuron: one excitatory and
one inhibitatory with the ratios obtained from the Mouse Brain Atlas.

## Install dependencies in a `conda` environment 

The Notebook is expected to be run in a `conda` environment,
in which `cerebrum` is installed. 
See the repo's [README.md](https://github.com/apache/airavata-cerebrum/blob/main/README.md) file for the instructions to
create the environment.

In [1]:
import os
import duckdb
import h5py
import numpy as np
import pandas as pd

from airavata_cerebrum.dataset.abc_mouse import (
    ABCDbMERFISH_CCFQuery,
    ABCDuckDBWriter,
    ExecParams,
    InitParams,
)

# Model Recipe

The Cerebrum Model Recipe defines key components for constructing a Network with Cerebrum. 

- **Data Providers:** Links to source database and query parameters
- **Filter and Transformers:** Data processing workflow such as filtering via selection criteria on the queried data.
- **Data2Model Mappers:** Mapper methods that map data to model components.
- **User Modifications:** Descriptions of user additions/removals to the purely data-driven model

The following figure shows the overall structure of a Model recipe definition.

<div>
<img src="img/Recipe.png" width="600"/>
</div>

In this example, we use **ONLY** use the **Data Provider** model. We use the data provider _'ABCDbMERFISH_CCFQuery'_ to acquire data from the [Mouse Brain Cell Atlas](https://alleninstitute.github.io/abc_atlas_access/descriptions/WMB_dataset.html).
Data from [Mouse Brain Cell Atlas](https://alleninstitute.github.io/abc_atlas_access/descriptions/WMB_dataset.html) is downloaded by  _'ABCDbMERFISH_CCFQuery'_  using the [abc_atlas_access](https://alleninstitute.github.io/abc_atlas_access/intro.html) package.

# Query the Brain Cell Atlas
## Input/Output/Query Parameters

The following python variables define the input/output file names and query parameters.
SEL_REGION and SEL_LAYER are the query parameters that informs the data provider class
the region and layer of interest.

In [2]:
# Input/Output/Parameters
# Location to download the data 
DOWNLOAD_BASE = "./cache/abc_mouse/"
OUT_DIR = "./"
# Location to save as a duckdb database
ABM_DB = "abm_mouse.db"

# Query Parameters
SEL_REGION = "VISp"
SEL_LAYER = "4"

## Create input/output directories
First, we create input and output directories if they don't exist

In [3]:
def make_dirs():
    if not os.path.exists(DOWNLOAD_BASE):
        os.makedirs(DOWNLOAD_BASE)
    if not os.path.exists(OUT_DIR):
        os.makedirs(OUT_DIR)

make_dirs()

## Download data
We run the data provider to dowload the summary data from  Cell Atlas and save locally as a duckdb database.

In [4]:
def download_abc_mouse(db_file: str = ABM_DB):
    qry: ABCDbMERFISH_CCFQuery = ABCDbMERFISH_CCFQuery(
        init_params=InitParams(download_base=DOWNLOAD_BASE)
    )
    qitr = qry.run(ExecParams(region=[SEL_REGION]), None, None)
    with duckdb.connect(db_file) as conn:
        db_writer = ABCDuckDBWriter(conn)
        db_writer.write(qitr)

download_abc_mouse()


## Explore the data from Mouse Atlas Database

 The Regional Neuron Distributions acquired from the Mouse Brain Atlas can be queried from the saved duckdb database as below.

In [5]:
with duckdb.connect(ABM_DB) as conn:
    rdf = conn.sql("SELECT * FROM abm_mouse").to_df()

rdf

,Region,Layer,nregion,inhibitory fraction,fraction wi. region,Vip fraction,Pvalb fraction,Sst fraction,Lamp5 fraction,Sst-Chodl fraction,IT-ENT fraction,IT-CTX fraction,IT-Other fraction,ET fraction,CT fraction,NP fraction,IT fraction
0,VISp,1,61884,0.405155,0.069663,0.147583,0.038168,0.035623,0.720102,0.007634,0.001733,0.986135,0.001733,0.000000,0.000000,0.000000,0.989601
1,VISp,2/3,61884,0.078755,0.348620,0.316690,0.332382,0.161912,0.158345,0.002140,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000
2,VISp,4,61884,0.111916,0.209699,0.131933,0.532773,0.315126,0.018487,0.000000,0.000000,0.962618,0.000000,0.029334,0.000212,0.007413,0.962618
3,VISp,5,61884,0.165428,0.177606,0.021739,0.487681,0.455797,0.022464,0.006522,0.001867,0.519247,0.000144,0.323039,0.043953,0.104137,0.521258
4,VISp,6a,61884,0.062779,0.163661,0.038321,0.562044,0.348540,0.025547,0.010949,0.001711,0.212443,0.001834,0.000611,0.759320,0.005378,0.215988
5,VISp,6b,61884,0.052910,0.030751,0.037500,0.475000,0.275000,0.087500,0.075000,0.000000,0.042598,0.004190,0.000000,0.725559,0.000000,0.046788


# Build the Model in SONATA
## Model Size Parameters

Parameters for number of nodes and edges, followed by random number generator.

In [6]:
#
N_NRN = 1000
AVG_DEGREE = 10  # average outgoing edges per neuron
N_EDGES = N_NRN * AVG_DEGREE
ID_RANGE = np.arange(N_NRN, dtype=np.int64)
RNG = np.random.default_rng(42)

## Estimate of Neuron counts with Atlas Data

Give the total number of neurons, the following code estiamtes the number of
excitatory and inhibitory neurons based on the ratio of cells found in
the MERFISH dataset in the Mouse Brain Atlas.

In [7]:
def neuron_counts(
    n: int = N_NRN, db_file: str = ABM_DB, layer: str = SEL_LAYER
) -> tuple[int, int]:
    r_sql = 'select "inhibitory fraction" from abm_mouse where layer='
    r_sql += f"'{layer}'"
    with duckdb.connect(db_file) as dconn:
        qrdf = dconn.sql(r_sql).to_df()
        inh_fraction = qrdf.iloc[0, 0]
        inh_neurons = int(n * inh_fraction)
        exc_neurons = n - inh_neurons
        return (exc_neurons, inh_neurons)

n_exc, n_inh = neuron_counts()

## Build SONATA File
We save the model to a SONTATA format as follows. 
SONATA format has four files:
1. Nodes file
2. Node types file
3. Edges file
4. Edge types file

### Nodes files

Node file is a HDF5 listing all the nodes, its positions and other properties.
In this simple example, we randomly generate the positions as below.

In [8]:
# Node positions
RND_POSITIONS = RNG.uniform(0, 1.0, size=(N_NRN, 3)).astype(np.float32)
NODE_POPULATION = "default"

def nodes_file(n_exc: int, n_inh: int):
    assert n_inh + n_exc == N_NRN
    # ── Write nodes.h5
    with h5py.File(f"{OUT_DIR}/nodes.h5", "w") as f:
        f.attrs["magic"] = np.uint32(0x0A7A)
        f.attrs["version"] = np.array([0, 1], dtype=np.uint32)
        pop = f.require_group(f"nodes/{NODE_POPULATION}")
        pop.create_dataset("node_id", data=ID_RANGE)
        pop.create_dataset(
            "node_type_id", data=np.array([100] * n_inh + [200] * n_exc, dtype=np.int64)
        )
        pop.create_dataset("node_group_id", data=ID_RANGE)
        pop.create_dataset("node_group_index", data=ID_RANGE)
        grp = pop.require_group("0")
        grp.create_dataset("x", data=RND_POSITIONS[:, 0])
        grp.create_dataset("y", data=RND_POSITIONS[:, 1])
        grp.create_dataset("z", data=RND_POSITIONS[:, 2])


nodes_file(n_exc, n_inh)

### Node types file

Node types file is a csv file containing all metadata, with one row each for each node type

In [ ]:
def node_types_file():
    node_type_rows = [
        dict(
            node_type_id=100,
            population=NODE_POPULATION,
            model_type="point_neuron",
            model_template="nrn:IntFire1",
            dynamics_params="inhibitory_params.json",
            cell_class="inhibitory",
            ei="i",
        ),
        dict(
            node_type_id=200,
            population=NODE_POPULATION,
            model_type="point_neuron",
            model_template="nrn:IntFire1",
            dynamics_params="excitatory_params.json",
            cell_class="excitatory",
            ei="e",
        ),
    ]
    pd.DataFrame(node_type_rows).to_csv(
        f"{OUT_DIR}/node_types.csv", sep=" ", index=False
    )

node_types_file()

### Edges files

Edge file is a HDF5 file, containing datasets that store all the edges and its properties.
Here, we use the random number generator to generate connections.

In [9]:
EDGE_POPULATION = f"{NODE_POPULATION}_to_{NODE_POPULATION}"
S_RAW, T_RAW = (
    RNG.integers(0, N_NRN, size=N_EDGES).astype(np.int64),
    RNG.integers(0, N_NRN, size=N_EDGES).astype(np.int64),
)
# Edge SRC/TGT ids
EDGE_SRC_ID, EDGE_TGT_ID = S_RAW[S_RAW != T_RAW], T_RAW[S_RAW != T_RAW]
# Edge Syn Weights
SYN_WEIGHTS = RNG.uniform(0.0, 1.0, size=N_EDGES).astype(np.float32)
DELAYS = RNG.uniform(0.5, 5.0, size=N_EDGES).astype(np.float32)


In [11]:
def edges_file(n_exc: int, n_inh: int):
    assert n_inh + n_exc == N_NRN
    src_e = EDGE_SRC_ID < n_exc
    tgt_e = EDGE_TGT_ID < n_exc
    edge_type_ids = np.where(
        src_e & tgt_e,
        10,
        np.where(src_e & ~tgt_e, 11, np.where(~src_e & tgt_e, 20, 21)),
    ).astype(np.int64)
    # Write edges.h5
    with h5py.File(f"{OUT_DIR}/edges.h5", "w") as f:
        f.attrs["magic"] = np.uint32(0x0A7A)
        f.attrs["version"] = np.array([0, 1], dtype=np.uint32)
        pop = f.require_group(f"edges/{EDGE_POPULATION}")
        ds = pop.create_dataset("source_node_id", data=EDGE_SRC_ID)
        ds.attrs["node_population"] = NODE_POPULATION
        ds = pop.create_dataset("target_node_id", data=EDGE_TGT_ID)
        ds.attrs["node_population"] = NODE_POPULATION
        pop.create_dataset("edge_type_id", data=edge_type_ids)
        pop.create_dataset(
            "edge_group_id", data=np.zeros(len(EDGE_SRC_ID), dtype=np.int64)
        )
        pop.create_dataset(
            "edge_group_index", data=np.arange(len(EDGE_SRC_ID), dtype=np.int64)
        )
        # Edge Syn Weights
        grp = pop.require_group("0")
        grp.create_dataset("syn_weight", data=SYN_WEIGHTS)
        grp.create_dataset("delay", data=DELAYS)


edges_file(n_exc, n_inh)

### Edge Types files

Edge types file is a csv file containing all metadata, one row each for each node type. 
Here we have four edge types for each of the possible pairing of excitatory and inhibitatory neurons

In [12]:
def edge_types_file():
    edge_type_rows = [
        dict(
            edge_type_id=10,
            population=EDGE_POPULATION,
            model_template="Exp2Syn",
            dynamics_params="AMPA_ExcToExc.json",
            connection_type="ExcToExc",
        ),
        dict(
            edge_type_id=11,
            population=EDGE_POPULATION,
            model_template="Exp2Syn",
            dynamics_params="AMPA_ExcToInh.json",
            connection_type="ExcToInh",
        ),
        dict(
            edge_type_id=20,
            population=EDGE_POPULATION,
            model_template="Exp2Syn",
            dynamics_params="GABA_InhToExc.json",
            connection_type="InhToExc",
        ),
        dict(
            edge_type_id=21,
            population=EDGE_POPULATION,
            model_template="Exp2Syn",
            dynamics_params="GABA_InhToInh.json",
            connection_type="InhToInh",
        ),
    ]
    # Write edge_types.csv
    pd.DataFrame(edge_type_rows).to_csv(
        f"{OUT_DIR}/edge_types.csv", sep=" ", index=False
    )

edge_types_file()